# Imputation Forward

Ce notebook :
1. Charge la base `base_avec_statut.parquet` depuis MinIO (couche Gold)
2. Identifie les agents hors barème et compte les mois incohérents
3. Applique une **imputation forward** (dernière valeur valide connue) sous conditions
4. Produit des statistiques descriptives et un export Excel


In [66]:
import io
import os
import polars as pl
import urllib3
from dotenv import load_dotenv
import pandas as pd
import boto3
from botocore.client import Config

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
load_dotenv(".env")

s3 = boto3.client(
    "s3",
    endpoint_url          = os.getenv("MINIO_ENDPOINT", "http://192.168.1.230:30137"),
    aws_access_key_id     = os.getenv("MINIO_ACCESS_KEY", "datalab-team"),
    aws_secret_access_key = os.getenv("MINIO_SECRET_KEY", "minio-datalabteam123"),
    config                = Config(signature_version="s3v4"),
    region_name           = "us-east-1",
    verify                = False,
)

buf = io.BytesIO()
s3.download_fileobj("gold", "panel_admin/base_avec_statut.parquet", buf)
buf.seek(0)
base = pl.read_parquet(buf)

print(f"Base chargée : {base.shape[0]:,} lignes × {base.shape[1]} colonnes")
print(f"Agents uniques : {base.select('matricule').n_unique():,}")
base.head(3)

Base chargée : 6,914,095 lignes × 55 colonnes
Agents uniques : 311,649


cle_unique,matricule,nom,sexe,date_naissance,situation_matrimoniale,nombre_enfant,situation_normalisee,CODE_ORGANISME,organisme,CODE_EMPLOI,emploi,Code_CITP,Metier_CITP,CODE_GRADE,grade,statut_fonctionnaire,CODE_FONCTION,fonction,CODE_SERVICE,service,CODE_AFFECTATION,lieu_affectation,CODE_POSTE,prise_service,date_retraite,age_retraite,GRADE,GRADE_SOURCE,mois_annee,annee,mois,montant_brut,montant_net,retenue_pension,impot,charge_patronale,montant_brut_SOURCE,montant_net_SOURCE,fichier_source,NB_POSTES,montant_brut_total,montant_brut_bareme,montant_net_total,ANNEE,MOIS_NUM,MOIS,SALAIRE_BRUT,SEXE_STD,bareme,grade_ref,bareme_detail,min_bareme,max_bareme,Statut
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,i32,f64,f64,f64,f64,f64,str,str,str,u32,f64,f64,f64,i32,i32,str,f64,str,str,str,str,f64,f64,str
"""100604R18""","""100604R""","""EDI ASSIRO MATHIAS""","""Masculin""","""1959-07-26""","""Célibataire""","""0""","""en_activite""","""18""","""Min. d'Etat de l'Agriculture, …","""03DF""","""Ingénieur Agronome""","""2132""","""Conseillers en agriculture, fo…","""13""","""Classe Exceptionnelle 3ème Ech…","""Fonctionnaire Catégorie A4""","""063""","""Représentation Groupe 3""","""1250""","""Direction Régionale""","""2AA""","""Abengourou""",null,"""2003-01-02""",null,null,"""A4""","""OBSERVE""","""012024""",2024,1,732455.0,590594.0,40741.0,94116.0,81530.908764,"""OBSERVE""","""OBSERVE""","""012024.xlsx""",1,732455.0,632455.0,590594.0,2024,1,"""Janvier""",632455.0,"""Homme""","""Barême Général""","""A4""","""Barême Général""",340285.0,662454.0,"""Conforme au barème"""
"""100604R18""","""100604R""","""EDI ASSIRO MATHIAS""","""Masculin""","""1959-07-26""","""Célibataire""","""0""","""en_activite""","""18""","""Min. d'Etat de l'Agriculture, …","""03DF""","""Ingénieur Agronome""","""2132""","""Conseillers en agriculture, fo…","""13""","""Classe Exceptionnelle 3ème Ech…","""Fonctionnaire Catégorie A4""","""063""","""Représentation Groupe 3""","""1250""","""Direction Régionale""","""2AA""","""Abengourou""",null,"""2003-01-02""",null,null,"""A4""","""OBSERVE""","""022024""",2024,2,732455.0,590594.0,40741.0,94116.0,81530.908764,"""OBSERVE""","""OBSERVE""","""022024.xlsx""",1,732455.0,632455.0,590594.0,2024,2,"""Février""",632455.0,"""Homme""","""Barême Général""","""A4""","""Barême Général""",340285.0,662454.0,"""Conforme au barème"""
"""100604R18""","""100604R""","""EDI ASSIRO MATHIAS""","""Masculin""","""1959-07-26""","""Célibataire""","""0""","""en_activite""","""18""","""Min. d'Etat de l'Agriculture, …","""03DF""","""Ingénieur Agronome""","""2132""","""Conseillers en agriculture, fo…","""13""","""Classe Exceptionnelle 3ème Ech…","""Fonctionnaire Catégorie A4""","""063""","""Représentation Groupe 3""","""1250""","""Direction Régionale""","""2AA""","""Abengourou""",null,"""2003-01-02""",null,null,"""A4""","""OBSERVE""","""032024""",2024,3,732455.0,590594.0,40741.0,94116.0,81530.908764,"""OBSERVE""","""OBSERVE""","""032024.xlsx""",1,732455.0,632455.0,590594.0,2024,3,"""Mars""",632455.0,"""Homme""","""Barême Général""","""A4""","""Barême Général""",340285.0,662454.0,"""Conforme au barème"""


In [67]:
# ═══════════════════════════════════════════════════════
# TABULATION DE LA VARIABLE Statut
# ═══════════════════════════════════════════════════════

tab_statut = (
    base
    .group_by("Statut")
    .agg(
        pl.len().alias("effectif"),
        pl.col("matricule").n_unique().alias("nb_agents")
    )
    .with_columns([
        (pl.col("effectif") / pl.col("effectif").sum() * 100).round(2).alias("pct_lignes"),
        (pl.col("nb_agents") / pl.col("nb_agents").sum() * 100).round(2).alias("pct_agents"),
    ])
    .sort("effectif", descending=True)
)

print(f"Distribution de la variable Statut ({base.shape[0]:,} lignes, {base['matricule'].n_unique():,} agents)\n")
print(tab_statut)

Distribution de la variable Statut (6,914,095 lignes, 311,649 agents)

shape: (4, 5)
┌────────────────────────────────┬──────────┬───────────┬────────────┬────────────┐
│ Statut                         ┆ effectif ┆ nb_agents ┆ pct_lignes ┆ pct_agents │
│ ---                            ┆ ---      ┆ ---       ┆ ---        ┆ ---        │
│ str                            ┆ u32      ┆ u32       ┆ f64        ┆ f64        │
╞════════════════════════════════╪══════════╪═══════════╪════════════╪════════════╡
│ Conforme au barème             ┆ 6458499  ┆ 300473    ┆ 93.41      ┆ 67.99      │
│ Inférieur au minimum du barème ┆ 304807   ┆ 36303     ┆ 4.41       ┆ 8.21       │
│ Supérieur au maximum du barème ┆ 142795   ┆ 104754    ┆ 2.07       ┆ 23.7       │
│ Cluster non déterminé          ┆ 7994     ┆ 424       ┆ 0.12       ┆ 0.1        │
└────────────────────────────────┴──────────┴───────────┴────────────┴────────────┘


In [68]:
list(base.columns)

['cle_unique',
 'matricule',
 'nom',
 'sexe',
 'date_naissance',
 'situation_matrimoniale',
 'nombre_enfant',
 'situation_normalisee',
 'CODE_ORGANISME',
 'organisme',
 'CODE_EMPLOI',
 'emploi',
 'Code_CITP',
 'Metier_CITP',
 'CODE_GRADE',
 'grade',
 'statut_fonctionnaire',
 'CODE_FONCTION',
 'fonction',
 'CODE_SERVICE',
 'service',
 'CODE_AFFECTATION',
 'lieu_affectation',
 'CODE_POSTE',
 'prise_service',
 'date_retraite',
 'age_retraite',
 'GRADE',
 'GRADE_SOURCE',
 'mois_annee',
 'annee',
 'mois',
 'montant_brut',
 'montant_net',
 'retenue_pension',
 'impot',
 'charge_patronale',
 'montant_brut_SOURCE',
 'montant_net_SOURCE',
 'fichier_source',
 'NB_POSTES',
 'montant_brut_total',
 'montant_brut_bareme',
 'montant_net_total',
 'ANNEE',
 'MOIS_NUM',
 'MOIS',
 'SALAIRE_BRUT',
 'SEXE_STD',
 'bareme',
 'grade_ref',
 'bareme_detail',
 'min_bareme',
 'max_bareme',
 'Statut']

In [69]:
nb_lignes = base["Statut"].is_in([
    "Inférieur au minimum du barème",
    "Supérieur au maximum du barème"
]).sum()

nb_agents = (
    base.filter(pl.col("Statut").is_in([
        "Inférieur au minimum du barème",
        "Supérieur au maximum du barème"
    ]))
    .select("matricule").n_unique()
)

print(f"Total lignes hors barème : {nb_lignes:,}")
print(f"Total agents hors barème : {nb_agents:,}")

Total lignes hors barème : 447,602
Total agents hors barème : 116,678


## 1. Exploration des agents hors barème


In [70]:
statuts_incoherents = [
    "Inférieur au minimum du barème",
    "Supérieur au maximum du barème"
]

hors = base.filter(pl.col("Statut").is_in(statuts_incoherents))
print(f"Lignes hors barème : {len(hors):,} / {len(base):,} ({len(hors)/len(base)*100:.2f}%)")
print(f"Agents concernés  : {hors.select('matricule').n_unique():,}")
hors.head(5)

Lignes hors barème : 447,602 / 6,914,095 (6.47%)
Agents concernés  : 116,678


cle_unique,matricule,nom,sexe,date_naissance,situation_matrimoniale,nombre_enfant,situation_normalisee,CODE_ORGANISME,organisme,CODE_EMPLOI,emploi,Code_CITP,Metier_CITP,CODE_GRADE,grade,statut_fonctionnaire,CODE_FONCTION,fonction,CODE_SERVICE,service,CODE_AFFECTATION,lieu_affectation,CODE_POSTE,prise_service,date_retraite,age_retraite,GRADE,GRADE_SOURCE,mois_annee,annee,mois,montant_brut,montant_net,retenue_pension,impot,charge_patronale,montant_brut_SOURCE,montant_net_SOURCE,fichier_source,NB_POSTES,montant_brut_total,montant_brut_bareme,montant_net_total,ANNEE,MOIS_NUM,MOIS,SALAIRE_BRUT,SEXE_STD,bareme,grade_ref,bareme_detail,min_bareme,max_bareme,Statut
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,i32,f64,f64,f64,f64,f64,str,str,str,u32,f64,f64,f64,i32,i32,str,f64,str,str,str,str,f64,f64,str
"""108076E31""","""108076E""","""HEMA CATHERINE""","""Féminin""","""1959-05-26""","""Célibataire""","""0""","""en_activite""","""31""","""Min. des Eaux et Forêts""","""9CEU""","""INGENIEUR PRINCIPAL: EAUX ET F…","""2132""","""Conseillers en agriculture, fo…","""13""","""Classe Exceptionnelle 3ème Ech…","""Fonctionnaire Catégorie A5""",null,""" ""","""0311""","""D A A F""","""1AA""","""Abidjan""",null,"""1977-10-01""",null,null,"""A5""","""OBSERVE""","""022024""",2024,2,1.135876e6,786287.0,41312.0,95800.0,82673.594238,"""OBSERVE""","""OBSERVE""","""022024.xlsx""",1,1.135876e6,935876.0,786287.0,2024,2,"""Février""",935876.0,"""Femme""","""Barême Général""","""A5""","""Barême Général""",379078.0,687823.0,"""Supérieur au maximum du barème"""
"""110082J22""","""110082J""","""KOUAO BROU""","""Masculin""","""1958-12-31""","""Marié""","""0""","""en_activite""","""22""","""Min. de l'Education Nationale …","""25WU""","""Inspecteur d'Enseignement Prim…","""2351""","""Spécialistes des méthodes péda…","""12""","""Classe Exceptionnelle 2ème Ech…","""Fonctionnaire Catégorie A4""","""054""","""Inspecteur Primaire""","""22BU""","""IEP Bongouanou""","""3CA""","""Bongouanou""",null,"""1978-01-07""",null,null,"""A4""","""OBSERVE""","""082024""",2024,8,1.021375e6,770918.0,59976.0,138890.0,120024.0,"""OBSERVE""","""OBSERVE""","""082024.xlsx""",1,1.021375e6,999375.0,770918.0,2024,8,"""Août""",999375.0,"""Homme""","""ENS SEC & PRIM""","""A4""","""ENS SEC & PRIM_SEC""",457071.0,819511.0,"""Supérieur au maximum du barème"""
"""112035G12""","""112035G""","""YAO YAO ADOLPHE""","""Masculin""","""1961-02-11""","""Marié""","""0""","""en_activite""","""12""","""Ministere des Finances et du B…","""9AAB""","""ADMINISTRATEUR GENERAL SERV.FI…","""1112 | 1211""","""Hauts fonctionnaires | Directe…","""31""","""1ère Classe 1er Echelon""","""Fonctionnaire Catégorie A7""","""749""","""Régisseur 1 Milliard - gr 2""","""1216""","""D.G.T.C.P""","""4SA""","""Seguela""",null,"""1995-01-03""",null,null,"""A7""","""OBSERVE""","""012024""",2024,1,957592.0,821044.0,42200.0,87344.0,84450.660264,"""OBSERVE""","""OBSERVE""","""012024.xlsx""",1,957592.0,582592.0,821044.0,2024,1,"""Janvier""",582592.0,"""Homme""","""Barême Général""","""A7""","""Barême Général""",646210.0,750915.0,"""Inférieur au minimum du barème"""
"""112035G12""","""112035G""","""YAO YAO ADOLPHE""","""Masculin""","""1961-02-11""","""Marié""","""0""","""en_activite""","""12""","""Ministere des Finances et du B…","""9AAB""","""ADMINISTRATEUR GENERAL SERV.FI…","""1112 | 1211""","""Hauts fonctionnaires | Directe…","""32""","""1ère Classe 2ème Echelon""","""Fonctionnaire Catégorie A7""","""749""","""Régisseur 1 Milliard - gr 2""","""1216""","""D.G.T.C.P""","""4SA""","""Seguela""",null,"""1995-01-03""",null,null,"""A7""","""OBSERVE""","""032024""",2024,3,980411.0,837443.0,43853.0,92111.0,87758.644658,"""OBSERVE""","""OBSERVE""","""032024.xlsx""",1,980411.0,605411.0,837443.0,2024,3,"""Mars""",605411.0,"""Homme""","""Barême Général""","""A7""","""Barême Général""",646210.0,750915.0,"""Inférieur au minimum du barème"""
"""112035G12""","""112035G""","""

In [71]:
# Nombre de mois incohérents par agent (matricule)
mois_incoherents = (
    base.group_by("matricule")
    .agg(
        pl.col("Statut").is_in(statuts_incoherents).sum().alias("nombre_de_mois_incoherents")
    )
    .filter(pl.col("nombre_de_mois_incoherents") > 0)
)

nb_total_agents = base.select("matricule").n_unique()
nb_avec_incoherence = len(mois_incoherents)

print(f"Agents totaux                      : {nb_total_agents:,}")
print(f"Agents avec au moins 1 mois hors barème : {nb_avec_incoherence:,}  ({100*nb_avec_incoherence/nb_total_agents:.1f}%)")
mois_incoherents.head(5)

Agents totaux                      : 311,649
Agents avec au moins 1 mois hors barème : 116,678  (37.4%)


matricule,nombre_de_mois_incoherents
str,u32
"""346275P""",3
"""382483N""",1
"""395293U""",2
"""351846H""",1
"""905481R""",1


In [72]:
# Répartition : combien d'agents par nombre de mois incohérents
table_repartition = (
    mois_incoherents
    .group_by("nombre_de_mois_incoherents")
    .agg(pl.col("matricule").n_unique().alias("nb_agents"))
    .sort("nombre_de_mois_incoherents")
    .with_columns(
        (pl.col("nb_agents") * pl.col("nombre_de_mois_incoherents")).alias("nb_lignes_incoherentes")
    )
    .with_columns([
        (pl.col("nb_agents") / pl.col("nb_agents").sum() * 100).round(2).alias("pct_agents"),
        (pl.col("nb_lignes_incoherentes") / pl.col("nb_lignes_incoherentes").sum() * 100).round(2).alias("pct_lignes"),
    ])
)
print(table_repartition)

# Totaux de référence
total_lignes_incoherentes = table_repartition.select(pl.col("nb_lignes_incoherentes").sum()).item()
total_agents_incoherents  = table_repartition.select(pl.col("nb_agents").sum()).item()

# Total des lignes potentiellement imputables (1 à 5 mois)
total_imputable = table_repartition.filter(pl.col("nombre_de_mois_incoherents") <= 5)
n_lignes_imp = total_imputable.select(pl.col("nb_lignes_incoherentes").sum()).item()
n_agents_imp = total_imputable.select(pl.col("nb_agents").sum()).item()

print(f"\nTotal lignes imputables (1-5 mois) : {n_lignes_imp:,}  ({100*n_lignes_imp/total_lignes_incoherentes:.1f}% des lignes hors barème)")
print(f"Total agents imputables (1-5 mois) : {n_agents_imp:,}  ({100*n_agents_imp/total_agents_incoherents:.1f}% des agents hors barème)")

table_repartition.to_pandas().to_excel(
    "repartition_mois_incoherents.xlsx",
    index=False,
    engine="openpyxl"
)
print(f"\n✅ Fichier exporté : repartition_mois_incoherents.xlsx ({len(table_repartition):,} lignes)")

shape: (24, 5)
┌────────────────────────────┬───────────┬────────────────────────┬────────────┬────────────┐
│ nombre_de_mois_incoherents ┆ nb_agents ┆ nb_lignes_incoherentes ┆ pct_agents ┆ pct_lignes │
│ ---                        ┆ ---       ┆ ---                    ┆ ---        ┆ ---        │
│ u32                        ┆ u32       ┆ u32                    ┆ f64        ┆ f64        │
╞════════════════════════════╪═══════════╪════════════════════════╪════════════╪════════════╡
│ 1                          ┆ 65713     ┆ 65713                  ┆ 56.32      ┆ 14.68      │
│ 2                          ┆ 15023     ┆ 30046                  ┆ 12.88      ┆ 6.71       │
│ 3                          ┆ 5149      ┆ 15447                  ┆ 4.41       ┆ 3.45       │
│ 4                          ┆ 3309      ┆ 13236                  ┆ 2.84       ┆ 2.96       │
│ 5                          ┆ 3385      ┆ 16925                  ┆ 2.9        ┆ 3.78       │
│ …                          ┆ …         ┆ … 

## 2. Méthodologie d'imputation — Valeur conforme la plus proche

**Principe** : pour chaque mois hors barème, on remplace la valeur par la valeur **conforme au barème la plus proche dans le temps**, en cherchant dans les deux directions (passé et futur), pour le même individu avec le même grade, emploi et organisme.

**Conditions d'éligibilité** :
- Le nombre de mois hors barème par `matricule` est entre **1 et 5**
- La valeur de remplacement provient d'un mois où le **Statut = "Conforme au barème"** pour le même `matricule` avec le même GRADE, emploi et CODE_ORGANISME

**Recherche de la valeur la plus proche** :
1. Forward fill dans le groupe `(matricule, GRADE, emploi, CODE_ORGANISME)` → valeur conforme la plus récente **avant** le mois hors barème
2. Backward fill dans le même groupe → valeur conforme la plus proche **après** le mois hors barème
3. On compare les distances (en mois) et on retient la valeur **la plus proche**
4. En cas d'égalité, priorité à la valeur passée (forward)

> Aucune condition de stabilité globale sur la carrière n'est requise.

In [73]:
# ─────────────────────────────────────────────────────
# ÉTAPE 1 — Éligibilité : nombre de mois hors barème par matricule
#           Condition : entre 1 et 5 mois non conformes au total
# ─────────────────────────────────────────────────────

# Clé de tri chronologique : MMYYYY → YYYYMM
base = base.with_columns(
    (pl.col("mois_annee").str.slice(2, 4) + pl.col("mois_annee").str.slice(0, 2)).alias("_periode_sort")
)

# Compter les mois hors barème par matricule
nb_hors_matricule = (
    base.group_by("matricule")
    .agg(
        pl.col("Statut").is_in(statuts_incoherents).sum().alias("nb_hors_groupe")
    )
)

base = base.join(nb_hors_matricule, on="matricule", how="left")

# Éligibilité : entre 1 et 5 mois non conformes pour cet agent
base = base.with_columns(
    (
        (pl.col("nb_hors_groupe") >= 1) & (pl.col("nb_hors_groupe") <= 5)
    ).alias("eligible_groupe")
)

# Calculs agents
nb_total_incoh  = base.filter(pl.col("Statut").is_in(statuts_incoherents)).select("matricule").n_unique()
nb_eligible     = base.filter(pl.col("eligible_groupe") & pl.col("Statut").is_in(statuts_incoherents)).select("matricule").n_unique()
nb_non_eligible = nb_total_incoh - nb_eligible

# Calculs lignes
nb_lignes_total  = base.filter(pl.col("Statut").is_in(statuts_incoherents)).shape[0]
nb_lignes_elig   = base.filter(pl.col("eligible_groupe") & pl.col("Statut").is_in(statuts_incoherents)).shape[0]
nb_lignes_non    = nb_lignes_total - nb_lignes_elig

print(f"Agents hors barème          : {nb_total_incoh:,}   |   Lignes : {nb_lignes_total:,}")
print(f"  → éligibles  (1-5 mois)  : {nb_eligible:,}  ({100*nb_eligible/nb_total_incoh:.1f}%)   |   Lignes : {nb_lignes_elig:,}  ({100*nb_lignes_elig/nb_lignes_total:.1f}%)")
print(f"  → non éligibles (>5 mois): {nb_non_eligible:,}  ({100*nb_non_eligible/nb_total_incoh:.1f}%)   |   Lignes : {nb_lignes_non:,}  ({100*nb_lignes_non/nb_lignes_total:.1f}%)")

Agents hors barème          : 116,678   |   Lignes : 447,602
  → éligibles  (1-5 mois)  : 92,579  (79.3%)   |   Lignes : 141,367  (31.6%)
  → non éligibles (>5 mois): 24,099  (20.7%)   |   Lignes : 306,235  (68.4%)


In [74]:
GROUP_COLS = ["matricule", "GRADE", "emploi", "CODE_ORGANISME"]

# ── Valeur conforme + index mensuel (pour calcul de distance) ──────────
base = base.with_columns([
    pl.when(pl.col("Statut") == "Conforme au barème")
      .then(pl.col("montant_brut_bareme"))
      .otherwise(None)
      .alias("_val_conforme"),

    (pl.col("_periode_sort").str.slice(0, 4).cast(pl.Int32) * 12 +
     pl.col("_periode_sort").str.slice(4, 2).cast(pl.Int32))
    .alias("_month_idx"),
])

base = base.with_columns(
    pl.when(pl.col("Statut") == "Conforme au barème")
      .then(pl.col("_month_idx"))
      .otherwise(None)
      .alias("_midx_conforme")
)

# ── Tri chronologique ──────────────────────────────────────────────────
base = base.sort(GROUP_COLS + ["_periode_sort"])

# ── Forward fill (passé) ───────────────────────────────────────────────
base = base.with_columns([
    pl.col("_val_conforme").fill_null(strategy="forward").over(GROUP_COLS).alias("_val_fwd"),
    pl.col("_midx_conforme").fill_null(strategy="forward").over(GROUP_COLS).alias("_midx_fwd"),
])

# ── Backward fill (futur) ──────────────────────────────────────────────
base = base.with_columns([
    pl.col("_val_conforme").fill_null(strategy="backward").over(GROUP_COLS).alias("_val_bwd"),
    pl.col("_midx_conforme").fill_null(strategy="backward").over(GROUP_COLS).alias("_midx_bwd"),
])

# ── Distance en mois ───────────────────────────────────────────────────
base = base.with_columns([
    (pl.col("_month_idx") - pl.col("_midx_fwd")).abs().alias("_dist_fwd"),
    (pl.col("_midx_bwd") - pl.col("_month_idx")).abs().alias("_dist_bwd"),
])

# ── Valeur la plus proche ──────────────────────────────────────────────
base = base.with_columns(
    pl.when(
        pl.col("_val_fwd").is_not_null() & (
            pl.col("_val_bwd").is_null() |
            (pl.col("_dist_fwd") <= pl.col("_dist_bwd"))
        )
    )
    .then(pl.col("_val_fwd"))
    .when(pl.col("_val_bwd").is_not_null())
    .then(pl.col("_val_bwd"))
    .otherwise(None)
    .alias("_valeur_proche")
)

# ── Source (calculé AVANT d'écraser montant_brut_bareme) ──────────────
_condition_impute = (
    pl.col("eligible_groupe") &
    pl.col("Statut").is_in(statuts_incoherents) &
    pl.col("_valeur_proche").is_not_null() &
    (pl.col("_valeur_proche") != pl.col("montant_brut_bareme"))
)

base = base.with_columns([
    pl.when(_condition_impute).then(pl.lit("IMPUTE")).otherwise(pl.lit("OBSERVE")).alias("montant_brut_bareme_source"),
    pl.when(_condition_impute).then(pl.lit("IMPUTE")).otherwise(pl.lit("OBSERVE")).alias("Statut_source"),
])

# ── Écraser montant_brut_bareme pour les mois éligibles imputés ────────
base_corrigee = base.with_columns(
    pl.when(
        pl.col("eligible_groupe") &
        pl.col("Statut").is_in(statuts_incoherents) &
        pl.col("_valeur_proche").is_not_null()
    )
    .then(pl.col("_valeur_proche"))
    .otherwise(pl.col("montant_brut_bareme"))
    .alias("montant_brut_bareme")
)

# ── Nettoyage colonnes temporaires ────────────────────────────────────
_cols_temp = [
    "_val_conforme", "_midx_conforme", "_month_idx",
    "_val_fwd", "_midx_fwd", "_val_bwd", "_midx_bwd",
    "_dist_fwd", "_dist_bwd", "_valeur_proche", "_periode_sort"
]
base_corrigee = base_corrigee.drop([c for c in _cols_temp if c in base_corrigee.columns])

print("✅ Imputation terminée")
print(f"   Lignes totales : {len(base_corrigee):,}")

✅ Imputation terminée
   Lignes totales : 6,914,095


In [77]:
# ─────────────────────────────────────────────────────
# ÉTAPE 3 — Distribution montant_brut_bareme_source / Statut_source
# ─────────────────────────────────────────────────────

distrib = (
    base_corrigee.group_by("montant_brut_bareme_source")
    .agg(
        pl.len().alias("nb_lignes"),
        pl.col("matricule").n_unique().alias("nb_agents")
    )
    .with_columns([
        (pl.col("nb_lignes") / pl.col("nb_lignes").sum() * 100).round(2).alias("pct_lignes"),
        (pl.col("nb_agents") / pl.col("nb_agents").sum() * 100).round(2).alias("pct_agents"),
    ])
    .sort("nb_lignes", descending=True)
)

print("Distribution montant_brut_bareme_source :")
print(distrib)

Distribution montant_brut_bareme_source :
shape: (2, 5)
┌────────────────────────────┬───────────┬───────────┬────────────┬────────────┐
│ montant_brut_bareme_source ┆ nb_lignes ┆ nb_agents ┆ pct_lignes ┆ pct_agents │
│ ---                        ┆ ---       ┆ ---       ┆ ---        ┆ ---        │
│ str                        ┆ u32       ┆ u32       ┆ f64        ┆ f64        │
╞════════════════════════════╪═══════════╪═══════════╪════════════╪════════════╡
│ OBSERVE                    ┆ 6784400   ┆ 311649    ┆ 98.12      ┆ 77.77      │
│ IMPUTE                     ┆ 129695    ┆ 89070     ┆ 1.88       ┆ 22.23      │
└────────────────────────────┴───────────┴───────────┴────────────┴────────────┘


## 3. Vérifications post-imputation


In [78]:
# ═══════════════════════════════════════════════════════════════
# VÉRIFICATION 1 — Y a-t-il des agents hors barème sans valeur
#                  après imputation ?
# ═══════════════════════════════════════════════════════════════

null_hors_bareme = (
    base_corrigee
    .filter(
        pl.col("eligible_groupe") &
        pl.col("Statut").is_in(statuts_incoherents)
    )
    .select(
        pl.col("montant_brut_bareme").is_null().sum().alias("nb_sans_valeur"),
        pl.len().alias("nb_lignes_eligibles")
    )
)

nb_sans = null_hors_bareme["nb_sans_valeur"][0]
nb_tot  = null_hors_bareme["nb_lignes_eligibles"][0]

print(f"Lignes hors barème éligibles       : {nb_tot:,}")
print(f"  → imputées (valeur trouvée)      : {nb_tot - nb_sans:,}  ({100*(nb_tot-nb_sans)/nb_tot:.1f}%)")
print(f"  → sans valeur (aucun conforme")
print(f"    dans le groupe)                : {nb_sans:,}  ({100*nb_sans/nb_tot:.1f}%)")

Lignes hors barème éligibles       : 141,367
  → imputées (valeur trouvée)      : 141,367  (100.0%)
  → sans valeur (aucun conforme
    dans le groupe)                : 0  (0.0%)


In [79]:
# ═══════════════════════════════════════════════════════════════
# VÉRIFICATION 2 — Répartition OBSERVE / IMPUTE
#                  par type de mois (conforme vs hors barème)
# ═══════════════════════════════════════════════════════════════

bilan = (
    base_corrigee
    .with_columns(
        pl.col("Statut").is_in(statuts_incoherents).alias("est_hors_bareme")
    )
    .group_by(["est_hors_bareme", "montant_brut_bareme_source"])
    .agg(
        pl.len().alias("nb_lignes"),
        pl.col("matricule").n_unique().alias("nb_agents")
    )
    .with_columns(
        pl.when(pl.col("est_hors_bareme"))
          .then(pl.lit("Hors barème"))
          .otherwise(pl.lit("Conforme"))
          .alias("type_mois")
    )
    .drop("est_hors_bareme")
    .with_columns([
        (pl.col("nb_lignes") / pl.col("nb_lignes").sum() * 100).round(2).alias("pct_lignes"),
        (pl.col("nb_agents") / pl.col("nb_agents").sum() * 100).round(2).alias("pct_agents"),
    ])
    .select(["type_mois", "montant_brut_bareme_source", "nb_lignes", "pct_lignes", "nb_agents", "pct_agents"])
    .sort(["type_mois", "montant_brut_bareme_source"])
)

print(bilan)

shape: (3, 6)
┌─────────────┬────────────────────────────┬───────────┬────────────┬───────────┬────────────┐
│ type_mois   ┆ montant_brut_bareme_source ┆ nb_lignes ┆ pct_lignes ┆ nb_agents ┆ pct_agents │
│ ---         ┆ ---                        ┆ ---       ┆ ---        ┆ ---       ┆ ---        │
│ str         ┆ str                        ┆ u32       ┆ f64        ┆ u32       ┆ f64        │
╞═════════════╪════════════════════════════╪═══════════╪════════════╪═══════════╪════════════╡
│ Conforme    ┆ OBSERVE                    ┆ 6466493   ┆ 93.53      ┆ 300796    ┆ 72.03      │
│ Hors barème ┆ IMPUTE                     ┆ 129695    ┆ 1.88       ┆ 89070     ┆ 21.33      │
│ Hors barème ┆ OBSERVE                    ┆ 317907    ┆ 4.6        ┆ 27726     ┆ 6.64       │
└─────────────┴────────────────────────────┴───────────┴────────────┴───────────┴────────────┘


In [80]:
# ═══════════════════════════════════════════════════════════════
# VÉRIFICATION 3 — Aperçu des lignes imputées
# ═══════════════════════════════════════════════════════════════

apercu = (
    base_corrigee
    .filter(pl.col("montant_brut_bareme_source") == "IMPUTE")
    .select([
        "matricule", "mois_annee", "GRADE", "emploi", "CODE_ORGANISME",
        "Statut", "nb_hors_groupe", "montant_brut_bareme", "montant_brut_bareme_source"
    ])
    .sort(["matricule", "mois_annee"])
)

nb_total_hors = base_corrigee.filter(pl.col("Statut").is_in(statuts_incoherents)).shape[0]
nb_impute     = len(apercu)

print(f"Lignes hors barème totales : {nb_total_hors:,}")
print(f"  → imputées               : {nb_impute:,}  ({100*nb_impute/nb_total_hors:.1f}%)")
print(f"  → non imputées           : {nb_total_hors - nb_impute:,}  ({100*(nb_total_hors-nb_impute)/nb_total_hors:.1f}%)")
print()
print(apercu.head(10))

Lignes hors barème totales : 447,602
  → imputées               : 129,695  (29.0%)
  → non imputées           : 317,907  (71.0%)

shape: (10, 9)
┌───────────┬────────────┬───────┬────────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ matricule ┆ mois_annee ┆ GRADE ┆ emploi     ┆ … ┆ Statut     ┆ nb_hors_g ┆ montant_b ┆ montant_b │
│ ---       ┆ ---        ┆ ---   ┆ ---        ┆   ┆ ---        ┆ roupe     ┆ rut_barem ┆ rut_barem │
│ str       ┆ str        ┆ str   ┆ str        ┆   ┆ str        ┆ ---       ┆ e         ┆ e_source  │
│           ┆            ┆       ┆            ┆   ┆            ┆ u32       ┆ ---       ┆ ---       │
│           ┆            ┆       ┆            ┆   ┆            ┆           ┆ f64       ┆ str       │
╞═══════════╪════════════╪═══════╪════════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 108076E   ┆ 022024     ┆ A5    ┆ INGENIEUR  ┆ … ┆ Supérieur  ┆ 1         ┆ 695142.0  ┆ IMPUTE    │
│           ┆            ┆       ┆ PRINCIPAL: ┆

## 4. Statistiques descriptives


In [81]:
# ═══════════════════════════════════════════════════════
# STATISTIQUES GLOBALES
# ═══════════════════════════════════════════════════════

nb_lignes_total      = len(base_corrigee)
nb_agents_total      = base_corrigee.select("matricule").n_unique()
nb_lignes_incoher    = base_corrigee.filter(pl.col("Statut").is_in(statuts_incoherents)).shape[0]
nb_null_apres        = base_corrigee.filter(pl.col("montant_brut_bareme").is_null()).shape[0]

print("📊 RÉSUMÉ GLOBAL")
print(f"   Lignes totales             : {nb_lignes_total:,}")
print(f"   Agents totaux              : {nb_agents_total:,}")
print(f"   Lignes hors barème         : {nb_lignes_incoher:,}  ({100*nb_lignes_incoher/nb_lignes_total:.1f}%)")
print(f"   Null après imputation      : {nb_null_apres:,}  ({100*nb_null_apres/nb_lignes_total:.2f}%)")

# ═══════════════════════════════════════════════════════
# IMPACT DE L'IMPUTATION
# ═══════════════════════════════════════════════════════

nb_impute  = base_corrigee.filter(pl.col("montant_brut_bareme_source") == "IMPUTE").shape[0]
nb_observe = base_corrigee.filter(pl.col("montant_brut_bareme_source") == "OBSERVE").shape[0]

print(f"\n📈 IMPACT IMPUTATION (sur les lignes hors barème)")
print(f"   Imputées  : {nb_impute:,}  ({100*nb_impute/nb_lignes_incoher:.1f}% des lignes hors barème)")
print(f"   Observées : {nb_observe - (nb_lignes_total - nb_lignes_incoher):,}  ({100*(nb_observe - (nb_lignes_total - nb_lignes_incoher))/nb_lignes_incoher:.1f}% des lignes hors barème)")

# ═══════════════════════════════════════════════════════
# STATISTIQUES montant_brut_bareme (lignes imputées)
# ═══════════════════════════════════════════════════════

comparaison = base_corrigee.filter(
    pl.col("montant_brut_bareme_source") == "IMPUTE"
).select([
    pl.col("montant_brut_bareme").mean().alias("moyenne"),
    pl.col("montant_brut_bareme").median().alias("mediane"),
    pl.col("montant_brut_bareme").min().alias("min"),
    pl.col("montant_brut_bareme").max().alias("max"),
])

print("\n📉 STATISTIQUES montant_brut_bareme (lignes imputées)")
print(comparaison)

# ═══════════════════════════════════════════════════════
# RÉSUMÉ PAR AGENT
# ═══════════════════════════════════════════════════════

resume_agents = (
    base_corrigee
    .group_by("matricule")
    .agg([
        pl.col("nb_hors_groupe").max(),
        pl.col("eligible_groupe").any(),
        (pl.col("montant_brut_bareme_source") == "IMPUTE").sum().alias("nb_mois_imputes")
    ])
    .filter(pl.col("eligible_groupe"))
)

nb_agents_elig   = len(resume_agents)
nb_agents_impute = resume_agents.filter(pl.col("nb_mois_imputes") > 0).shape[0]

print(f"\n👤 RÉSUMÉ AGENTS ÉLIGIBLES")
print(f"   Agents éligibles             : {nb_agents_elig:,}")
print(f"   Agents effectivement imputés : {nb_agents_impute:,}  ({100*nb_agents_impute/nb_agents_elig:.1f}%)")
print(f"   Mois imputés total           : {resume_agents.select(pl.col('nb_mois_imputes').sum()).item():,}")

📊 RÉSUMÉ GLOBAL
   Lignes totales             : 6,914,095
   Agents totaux              : 311,649
   Lignes hors barème         : 447,602  (6.5%)
   Null après imputation      : 0  (0.00%)

📈 IMPACT IMPUTATION (sur les lignes hors barème)
   Imputées  : 129,695  (29.0% des lignes hors barème)
   Observées : 317,907  (71.0% des lignes hors barème)

📉 STATISTIQUES montant_brut_bareme (lignes imputées)
shape: (1, 4)
┌───────────────┬──────────┬──────────┬────────────┐
│ moyenne       ┆ mediane  ┆ min      ┆ max        │
│ ---           ┆ ---      ┆ ---      ┆ ---        │
│ f64           ┆ f64      ┆ f64      ┆ f64        │
╞═══════════════╪══════════╪══════════╪════════════╡
│ 421800.124978 ┆ 359632.0 ┆ 191417.0 ┆ 1.504153e6 │
└───────────────┴──────────┴──────────┴────────────┘

👤 RÉSUMÉ AGENTS ÉLIGIBLES
   Agents éligibles             : 92,579
   Agents effectivement imputés : 89,070  (96.2%)
   Mois imputés total           : 129,695


## 5. Export Excel


In [ ]:
# Export des lignes hors barème éligibles
export_df = (
    base_corrigee
    .filter(pl.col("eligible_groupe") & pl.col("Statut").is_in(statuts_incoherents))
    .select([
        "matricule", "nom", "GRADE", "emploi", "CODE_ORGANISME", "organisme",
        "mois_annee", "annee", "mois", "Statut",
        "montant_brut_bareme", "montant_brut_bareme_source",
        "Statut_source",
        "min_bareme", "max_bareme", "nb_hors_groupe"
    ])
)

export_path = "resultats_imputation_forward.xlsx"
export_df.to_pandas().to_excel(export_path, index=False, engine="openpyxl")
print(f"✅ Fichier Excel exporté : {export_path} ({len(export_df):,} lignes)")